In [1]:
# Osteoporosis Detection — Pipeline
# Anuki Kithara
# Date: 28/04/2026
# Models: SVR, KNN, Random Forest

In [2]:
#1 - IMPORTS

In [1]:
import os
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Image I/O and feature extraction
import cv2
from skimage.feature import hog
from skimage.measure import moments_hu, moments_central, moments_normalized, moments

# Modelling
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    confusion_matrix, balanced_accuracy_score, classification_report
)

import warnings
warnings.filterwarnings("ignore")

# Same seed everywhere
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plot style
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

In [2]:
#2 - FILE PATHS

In [3]:
IMAGE_DIR  = Path("#")       # folder path
TRAIN_CSV  = Path("#")       # train csv path
VAL_CSV    = Path("#")       # validation csv path

TRAIN_IMAGE_DIR = Path("train_images")
VAL_IMAGE_DIR   = Path("val_images")

# Image and feature settings
IMG_SIZE = 128                # resize images to 128x128
HIST_BINS = 64                # grayscale histogram bins

# T-score reference
REF_BMD = 0.86
REF_SD  = 0.12

# Helper functions for converting BMD to T-score

def t_score(bmd):
    return (bmd - REF_BMD) / REF_SD

def to_binary_label(t):
    return "Normal" if t >= -1 else "Low BMD"

# Checks if the files exist
assert TRAIN_CSV.exists(), "Training CSV not found"
assert VAL_CSV.exists(), "Validation CSV not found"
assert IMAGE_DIR.exists(), "Image folder not found"

In [4]:
#3 - LOAD METADATA

In [5]:
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)

# Checks dimensions and columns
print(f"Train: {train_df.shape[0]} rows, {train_df.shape[1]} columns")
print(f"Val:   {val_df.shape[0]} rows, {val_df.shape[1]} columns")
train_df.head()

Train: 377 rows, 4 columns
Val:   54 rows, 4 columns


,image,interview_age,Gender,BMD
0,6.E.1_9021102_20090706_001.png,888,F,0.696443
1,6.E.1_9030718_20090824_001.png,960,F,1.201649
2,6.E.1_9047519_20090720_001.png,876,F,0.945649
3,6.C.1_9048789_20090226_001.png,780,F,0.927137
4,6.E.1_9054972_20090911_001.png,828,F,0.987368


In [6]:
#4 - ORGANISE IMAGES INTO FOLDERS

In [8]:
# Uncomment and run once to organise images into train and val folders
# Then comment this block again.

import shutil

def organise_split(source_dir, dest_dir, df):
    dest_dir.mkdir(exist_ok=True)
    
    for filename in df["image"]:
        src = source_dir / filename
        dst = dest_dir / filename
        
        if src.exists() and not dst.exists():
            shutil.copy2(src, dst)


organise_split(IMAGE_DIR, TRAIN_IMAGE_DIR, train_df)
organise_split(IMAGE_DIR, VAL_IMAGE_DIR, val_df)

print("Image folders organised.")

In [9]:
#5 - EDA

In [10]:
#INSERT EDA CODE HERE

In [ ]:
#6 - PREPROCESSING

In [ ]:
#INSERT PREPROCESSING CODE HERE

In [ ]:
#7 - COMBINE IMAGE FEATURES AND METADATA

In [ ]:
X_train = np.hstack([X_img_train, X_meta_train])
X_val   = np.hstack([X_img_val,   X_meta_val])
y_train = train_df["BMD"].values
y_val   = val_df["BMD"].values

print(f"Combined train shape: {X_train.shape}")
print(f"Combined val shape:   {X_val.shape}")
print(f"y_train shape:        {y_train.shape}")

In [ ]:
#8 - PIPELINE SCALING

In [ ]:
pipeline = Pipeline([
    # Standardise features
    ("scaler", StandardScaler()),
    # Reduce high-dimensional image features
    ("pca",    PCA(n_components=50, random_state=RANDOM_STATE)),
    #change this to your own model
    ("model",    SVR()),
])

param_grid = {
    "pca__n_components": [50, 100],
    #these are svr specific parameters. swap for your model
    "model__kernel":       ["rbf", "linear"],
    "model__C":            [0.1, 1, 10, 100],
    "model__epsilon":      [0.01, 0.05, 0.1],
    "model__gamma":        ["scale", "auto"],
}

#5 fold cross validation
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

#grid search
#gridSearchCV tries every parameter combination above and chooses the combination with the best cross-validation MAE.
#sklearn uses negative MAE because it maximises scores which we then convert back to positive MAE when printing
search = GridSearchCV(
    pipeline,
    param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    refit=True,       # after picking best params, refit on full train set
)

#model training
search.fit(X_train, y_train)

print("\n=== Best CV result ===")
print(f"Best CV MAE: {-search.best_score_:.4f}")
print(f"Best params: {search.best_params_}")

In [ ]:
#9 - REGRESSION EVALUATION

In [ ]:
best_model = search.best_estimator_

y_pred_train = best_model.predict(X_train)
y_pred_val   = best_model.predict(X_val)

def regression_report(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_true, y_pred)
    print(f"--- {label} ---")
    print(f"  MAE  = {mae:.4f}")
    print(f"  MSE  = {mse:.6f}")
    print(f"  RMSE = {rmse:.4f}")
    print(f"  R²   = {r2:.4f}")
    return {"MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2}

train_metrics = regression_report(y_train, y_pred_train, "Training set")
print()
val_metrics   = regression_report(y_val,   y_pred_val,   "Validation set")

In [ ]:
#10 - DIAGNOSTIC PLOTS

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Predicted vs actual on validation
axes[0].scatter(y_val, y_pred_val, alpha=0.7, color="steelblue", edgecolor="white")
lo, hi = min(y_val.min(), y_pred_val.min()), max(y_val.max(), y_pred_val.max())
axes[0].plot([lo, hi], [lo, hi], "r--", label="Perfect prediction")
axes[0].set_xlabel("Actual BMD (g/cm²)")
axes[0].set_ylabel("Predicted BMD (g/cm²)")
axes[0].set_title("Validation: predicted vs actual")
axes[0].legend()

# Residual plot
residuals = y_val - y_pred_val
axes[1].scatter(y_pred_val, residuals, alpha=0.7, color="darkorange", edgecolor="white")
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_xlabel("Predicted BMD (g/cm²)")
axes[1].set_ylabel("Residual (actual − predicted)")
axes[1].set_title("Validation: residuals")

plt.tight_layout()
plt.show()

In [ ]:
#11 - CONVERT PREDICTED BMD

In [ ]:
# Convert predicted BMD into T-scores and derive Normal vs Low BMD labels

T_pred_val = t_score(y_pred_val)
T_true_val = t_score(y_val)

label_pred_val = pd.Series(T_pred_val).apply(to_binary_label)
label_true_val = pd.Series(T_true_val).apply(to_binary_label)

# Show side by side
results_df = pd.DataFrame({
    "filename":      val_df["image"].values,
    "true_BMD":      y_val.round(4),
    "pred_BMD":      y_pred_val.round(4),
    "true_T":        T_true_val.round(3),
    "pred_T":        T_pred_val.round(3),
    "true_label":    label_true_val.values,
    "pred_label":    label_pred_val.values,
})
results_df.head(15)

In [ ]:
#12 - EVALUATION

In [ ]:
#Confusion Matrix

labels_order = ["Normal", "Low BMD"]
cm = confusion_matrix(label_true_val, label_pred_val, labels=labels_order)

# Print confusion matrix
print("Confusion matrix (rows = true, cols = predicted):")
print(pd.DataFrame(cm, index=labels_order, columns=labels_order))

# Sensitivity and specificity
TN, FP, FN, TP = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]
sensitivity = TP / (TP + FN) if (TP + FN) > 0 else float("nan")
specificity = TN / (TN + FP) if (TN + FP) > 0 else float("nan")
bal_acc     = balanced_accuracy_score(label_true_val, label_pred_val)

print(f"\nSensitivity (Low BMD recall):  {sensitivity:.3f}")
print(f"Specificity (Normal recall):   {specificity:.3f}")
print(f"Balanced accuracy:             {bal_acc:.3f}")

print("\nFull classification report:")
print(classification_report(label_true_val, label_pred_val, labels=labels_order, zero_division=0))

# Heatmap visualisation
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels_order, yticklabels=labels_order, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Validation confusion matrix")
plt.tight_layout(); plt.show()